# PHASE 1 — QLoRA trên SYNTHETIC ĐA DẠNG (v2), eval sạch n=100 (Kaggle T4)
**Mục tiêu:** kiểm giả thuyết "synthetic chân thực hơn (pools+gen_synthetic đã diversify) có tổng quát hoá tốt hơn bản cũ 0.2832 và tiến sát/vượt rule không".
- Train **100% trên note synthetic sinh mới** (KHÔNG đụng file test) -> cả 100 file test (dev/gold) đều là held-out -> đo tổng quát hoá THẬT trên n=100.
- Mốc so: **rule** (chấm trên cùng 100 file) và synthetic-only cũ 0.2832.

Settings: **GPU T4 x2**, **Internet On**. Chạy nền: **Save Version -> Save & Run All (Commit)**. ~6-8h.

In [ ]:
# 1) Code - nhanh Duckun
%cd /kaggle/working
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /kaggle/working/viettel-medreason
!git fetch -q origin Duckun && git checkout -q Duckun && git pull -q origin Duckun
!git log --oneline -1

In [ ]:
# 2) Env - giu numpy/torch/transformers ban Kaggle; chi update bnb/peft/accelerate
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
import numpy, torch
print('numpy',numpy.__version__,'| torch',torch.__version__,'| GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not numpy.__version__.startswith('2'):
    raise SystemExit('numpy != 2.x -> Restart & Run All')

In [ ]:
# 3) Sinh SYNTHETIC DA DANG (v2) - 800 note, seed co dinh (reproducible, KHONG dung test)
!python src/datagen/gen_synthetic.py --n 800 --seed 42
import json
n=sum(1 for _ in open('data/synthetic/train_sft.jsonl',encoding='utf-8'))
print('so mau SFT:', n)

In [ ]:
# 4) SMOKE (kiem khong OOM o max_len 2560)
import os; os.environ['CUDA_VISIBLE_DEVICES']='0'
MODEL='Qwen/Qwen2.5-3B-Instruct'
!python scripts/train_qlora.py --data data/synthetic/train_sft.jsonl --model {MODEL} --out /kaggle/working/smoke --epochs 0.05 --max-len 2560 --seed 42

In [ ]:
# 5) TRAIN synthetic-only - 800 mau x2 epoch, max_len 2560. ~6-8h T4.
ADAPTER='/kaggle/working/qwen-lora-synth'
MODEL='Qwen/Qwen2.5-3B-Instruct'
!python scripts/train_qlora.py --data data/synthetic/train_sft.jsonl --model {MODEL} --out {ADAPTER} --epochs 2 --batch 1 --grad-accum 8 --lr 2e-4 --max-len 2560 --seed 42
!ls -la {ADAPTER}

In [ ]:
# 6) Config -> backend llm + adapter synthetic
import yaml
ADAPTER='/kaggle/working/qwen-lora-synth'
cfg=yaml.safe_load(open('configs/config.yaml',encoding='utf-8'))
cfg['extract']['backend']='llm'
cfg['extract']['lora_adapter']=ADAPTER
cfg['extract']['llm_model']='Qwen/Qwen2.5-3B-Instruct'
yaml.safe_dump(cfg,open('configs/config.yaml','w',encoding='utf-8'),allow_unicode=True,sort_keys=False)
print('adapter =',cfg['extract']['lora_adapter'])

In [ ]:
# 7) Du doan 100 file: QLoRA (llm) + rule (moc so)
!python src/pipeline.py --input data/test/input --output dev_pred  --backend llm
!python src/pipeline.py --input data/test/input --output rule_pred --backend rule

In [ ]:
# 8) GO/NO-GO - eval SACH tren ca 100 file (train chi tren synthetic sinh moi)
print('===== QLoRA synthetic-v2 | n=100 (tong quat hoa THAT) =====')
!python src/eval/official_scorer.py --pred dev_pred --gold data/dev/gold
print(); print('===== RULE | n=100 (moc so cung tap) =====')
!python src/eval/official_scorer.py --pred rule_pred --gold data/dev/gold
print()
print('>> QLoRA > RULE => synthetic da dang thang THAT, dang scale.')
print('>> ket giua 0.28 va rule => diversify co tac dung nhung chua du, sang Phase 2 (LLM sinh note).')
print('>> ~0.28 => synthetic khong phai duong, don Phase 2 / linking.')